# Knowledge Distillation: Alpha & Temperature Grid Search — Part 3: 4-Layer-Standard Runs (1 of 2)

This notebook continues the hyperparameter grid search.
It runs 5 experiments and merges results with previously completed runs for final analysis.


In [ ]:
!pip install transformers datasets accelerate scikit-learn einops

# Configuration & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import gc
import os
import json
import time
import sys
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BertConfig,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback,
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# ===== Configurable Constants =====
TEACHER_NOTEBOOK_SLUG = "ml-project-kaggle15916a635e"
TEACHER_CHECKPOINT_PATH = f"/kaggle/input/{TEACHER_NOTEBOOK_SLUG}/lr_2e-05_results/checkpoint-4375"  # Best: 96.71%
TEACHER_FALLBACK_PATH = f"/kaggle/input/{TEACHER_NOTEBOOK_SLUG}/human_promoter_model"                # Fallback: 96.40%
MODEL_NAME = "quietflamingo/dnabert2-no-flashattention"
MAX_LENGTH = 300
BATCH_SIZE = 16
NUM_EPOCHS = 10
SEED = 42

# ===== Hyperparameter Grid =====
ALPHAS = [0.3, 0.5, 0.7]
TEMPERATURES = [1.0, 3.0, 5.0]

# ===== Reproducibility =====
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ===== GPU Info =====
device = "cuda" if torch.cuda.is_available() else "cpu"
num_gpus = torch.cuda.device_count()
print(f"Device: {device}")
print(f"Number of GPUs: {num_gpus}")
for i in range(num_gpus):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {name} ({mem:.1f} GB)")

print(f"\nHyperparameter grid: {len(ALPHAS)} alphas x {len(TEMPERATURES)} temps = {len(ALPHAS)*len(TEMPERATURES)} combos per architecture")
print(f"Alphas: {ALPHAS}")
print(f"Temperatures: {TEMPERATURES}")

# Load Dataset & Tokenize

In [ ]:
# Load HUMAN promoter dataset from InstaDeepAI
print("Loading human promoter dataset from InstaDeepAI...")
raw_dataset = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks")

# Filter for promoter_all task only
def filter_promoter_all(example):
    return example["task"] == "promoter_all"

print("Filtering for promoter_all task...")
train_data_raw = raw_dataset["train"].filter(filter_promoter_all)
test_data_raw = raw_dataset["test"].filter(filter_promoter_all)

# Prepare data
def prepare_data(example):
    return {
        "sequence": example["sequence"].upper(),
        "label": int(example["label"]),
    }

train_data = train_data_raw.map(prepare_data)
test_data = test_data_raw.map(prepare_data)

print(f"Training samples: {len(train_data)}, Test samples: {len(test_data)}")

# Tokenize
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

def tokenize_func(examples):
    return tokenizer(examples["sequence"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

print(f"Tokenizing with max_length={MAX_LENGTH}...")
tokenized_train = train_data.map(tokenize_func, batched=True)
tokenized_test = test_data.map(tokenize_func, batched=True)

# Set format for PyTorch and rename label -> labels for HuggingFace Trainer
tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
tokenized_test.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")

train_dataset = tokenized_train
eval_dataset = tokenized_test

print(f"Tokenization complete!")
print(f"Train columns: {train_dataset.column_names}")
print(f"Sample input_ids shape: {train_dataset[0]['input_ids'].shape}")

# Load Teacher Model

In [ ]:
# Auto-discover teacher model path under /kaggle/input/
# Kaggle mounts notebook outputs at varying depths, e.g.:
#   /kaggle/input/notebooks/username/slug/lr_2e-05_results/checkpoint-4375/
import glob
from safetensors.torch import load_file as load_safetensors
from transformers import AutoConfig
from transformers.dynamic_module_utils import get_class_from_dynamic_module

def find_teacher_model():
    """Search all Kaggle inputs (recursively) for teacher checkpoint files."""
    base = "/kaggle/input"
    
    # Priority 1: Best checkpoint (96.71%)
    for filename in ["model.safetensors", "pytorch_model.bin"]:
        matches = glob.glob(f"{base}/**/lr_2e-05_results/checkpoint-4375/{filename}", recursive=True)
        if matches:
            return os.path.dirname(matches[0]), "best checkpoint (96.71%)"
    
    # Priority 2: Phase 1 model (96.40%)
    for filename in ["model.safetensors", "pytorch_model.bin"]:
        matches = glob.glob(f"{base}/**/human_promoter_model/{filename}", recursive=True)
        if matches:
            return os.path.dirname(matches[0]), "Phase 1 model (96.40%)"
    
    # Priority 3: Any checkpoint with config.json
    matches = glob.glob(f"{base}/**/human_promoter_results/checkpoint-*/config.json", recursive=True)
    if matches:
        return os.path.dirname(matches[0]), "fallback checkpoint"
    
    return None, None

teacher_path, teacher_desc = find_teacher_model()

if teacher_path is None:
    print("Contents of /kaggle/input/:")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root.replace("/kaggle/input", "").count(os.sep)
        if depth < 3:
            indent = "  " * depth
            print(f"{indent}{os.path.basename(root)}/")
    raise FileNotFoundError(
        "Teacher model not found!\n"
        "Please add the teacher notebook output as a Kaggle input:\n"
        "  File -> Add Data -> Your Datasets -> Notebook Output"
    )

print(f"Found teacher: {teacher_desc}")
print(f"Path: {teacher_path}")

# Load config from hub and patch missing fields
config = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
config.num_labels = 2
config.pad_token_id = tokenizer.pad_token_id  # DNABERT-2 bert_layers.py:39 needs this

# Get DNABERT-2's custom model class from its auto_map
# (bypasses from_pretrained's meta-device context that breaks ALiBi tensor math)
class_ref = config.auto_map["AutoModelForSequenceClassification"]
model_class = get_class_from_dynamic_module(class_ref, MODEL_NAME)

# Direct instantiation on CPU — no meta device wrapping
teacher_model = model_class(config)

# Load fine-tuned checkpoint weights
weights_file = os.path.join(teacher_path, "model.safetensors")
if os.path.exists(weights_file):
    state_dict = load_safetensors(weights_file)
else:
    weights_file = os.path.join(teacher_path, "pytorch_model.bin")
    state_dict = torch.load(weights_file, map_location="cpu", weights_only=True)

teacher_model.load_state_dict(state_dict)
print(f"Loaded fine-tuned weights from: {os.path.basename(weights_file)}")

teacher_model.eval()

teacher_params = sum(p.numel() for p in teacher_model.parameters())
print(f"Teacher model loaded successfully!")
print(f"Teacher parameters: {teacher_params:,} ({teacher_params/1e6:.1f}M)")

# Distillation Trainer & Student Architectures

In [ ]:
from transformers.modeling_outputs import SequenceClassifierOutput

# ===== DISTILLATION TRAINER =====
class DistillationTrainer(Trainer):
    def __init__(self, *args, teacher_model=None, alpha=0.5, temperature=3.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model.to(self.args.device)
        self.alpha = alpha
        self.temperature = temperature
        self.teacher_model.eval()

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # Student predictions
        outputs_student = model(**inputs)
        student_logits = outputs_student.logits
        labels = inputs.get("labels")

        # Teacher predictions (no gradients needed)
        with torch.no_grad():
            outputs_teacher = self.teacher_model(**inputs)
            teacher_logits = outputs_teacher.logits

        # Soft Loss: Matching the Teacher's probability distribution
        soft_loss = nn.KLDivLoss(reduction="batchmean")(
            F.log_softmax(student_logits / self.temperature, dim=-1),
            F.softmax(teacher_logits / self.temperature, dim=-1)
        ) * (self.temperature ** 2)

        # Hard Loss: Correctness against ground truth
        hard_loss = F.cross_entropy(student_logits, labels)

        # Final balanced loss
        loss = self.alpha * hard_loss + (1 - self.alpha) * soft_loss

        return (loss, outputs_student) if return_outputs else loss


# ===== HYBRID CNN-TRANSFORMER STUDENT =====
class DNAHybridStudent(nn.Module):
    def __init__(self, layers=2):
        super().__init__()
        # 1. Very small Transformer backbone
        config = BertConfig.from_pretrained("quietflamingo/dnabert2-no-flashattention", num_hidden_layers=layers)
        self.bert = BertForSequenceClassification(config).bert
        
        # 2. CNN Head for Motif Scanning
        self.conv = nn.Conv1d(in_channels=768, out_channels=128, kernel_size=9, padding=4)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.classifier = nn.Linear(128, 2)

    def forward(self, input_ids, attention_mask=None, labels=None):
        # Transformer pass
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state  # [batch, seq_len, 768]
        
        # CNN pass (permute to [batch, 768, seq_len])
        x = sequence_output.permute(0, 2, 1)
        x = torch.relu(self.conv(x))
        x = self.pool(x).squeeze(-1)
        
        logits = self.classifier(x)
        
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
            
        return SequenceClassifierOutput(loss=loss, logits=logits)


# ===== HELPER: PURE TRANSFORMER STUDENT =====
def get_pure_transformer(layers):
    config = BertConfig.from_pretrained(
        "quietflamingo/dnabert2-no-flashattention",
        num_hidden_layers=layers, num_labels=2,
    )
    return BertForSequenceClassification(config)


# ===== COMPUTE METRICS =====
def compute_metrics(pred):
    labels = pred.label_ids
    logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
    }


# ===== PROGRESS CALLBACK =====
class ProgressCallback(TrainerCallback):
    """Custom callback to force print output in Kaggle"""
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            print(f"Step {state.global_step}: {logs}", flush=True)

    def on_epoch_begin(self, args, state, control, **kwargs):
        print(f"\n{'='*60}", flush=True)
        print(f"Starting Epoch {int(state.epoch) + 1}/{args.num_train_epochs}", flush=True)
        print(f"{'='*60}", flush=True)

    def on_epoch_end(self, args, state, control, **kwargs):
        print(f"Epoch {int(state.epoch)} completed!", flush=True)


# ===== PARAMETER COUNTING =====
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


print("All components defined successfully!")

# Run Grid Search: 2 Architectures x Alpha x Temperature

In [ ]:
# ===== LOAD PREVIOUSLY COMPLETED RESULTS =====
# Try loading accumulated results from previous notebook outputs (Kaggle inputs)
import glob

_hardcoded_results = {
    "Hybrid-CNN-3Layer_a0.3_t1.0": {
        "architecture": "Hybrid-CNN-3Layer", "alpha": 0.3, "temperature": 1.0,
        "accuracy": 0.9282, "f1": 0.9273, "precision": 0.9393, "recall": 0.9155,
        "loss": 0.1052, "params": 26281346, "compression": 4.454, "train_time_min": 154.2
    },
    "Hybrid-CNN-3Layer_a0.3_t3.0": {
        "architecture": "Hybrid-CNN-3Layer", "alpha": 0.3, "temperature": 3.0,
        "accuracy": 0.9331, "f1": 0.9326, "precision": 0.9402, "recall": 0.9250,
        "loss": 0.1195, "params": 26281346, "compression": 4.454, "train_time_min": 154.3
    },
    "Hybrid-CNN-3Layer_a0.3_t5.0": {
        "architecture": "Hybrid-CNN-3Layer", "alpha": 0.3, "temperature": 5.0,
        "accuracy": 0.9309, "f1": 0.9302, "precision": 0.9400, "recall": 0.9206,
        "loss": 0.1226, "params": 26281346, "compression": 4.454, "train_time_min": 154.3
    },
    "Hybrid-CNN-3Layer_a0.5_t1.0": {
        "architecture": "Hybrid-CNN-3Layer", "alpha": 0.5, "temperature": 1.0,
        "accuracy": 0.9274, "f1": 0.9258, "precision": 0.9461, "recall": 0.9064,
        "loss": 0.1392, "params": 26281346, "compression": 4.454, "train_time_min": 154.1
    }
}

# Search for partial_results.json from any previous notebook output
_prev_files = glob.glob("/kaggle/input/**/partial_results.json", recursive=True)
if _prev_files:
    # Load the largest (most accumulated) results file
    _best_file = max(_prev_files, key=lambda f: os.path.getsize(f))
    with open(_best_file) as _f:
        final_results = json.load(_f)
    print(f"Loaded {len(final_results)} results from previous run: {_best_file}")
else:
    final_results = _hardcoded_results
    print(f"No previous results file found, using {len(final_results)} hardcoded results")

# Ensure hardcoded results are always present (merge)
for _k, _v in _hardcoded_results.items():
    if _k not in final_results:
        final_results[_k] = _v
        print(f"  Added missing hardcoded result: {_k}")


# ===== DEFINE ARCHITECTURES =====
architectures = {
    "Hybrid-CNN-3Layer": lambda: DNAHybridStudent(layers=3),
    "4-Layer-Standard": lambda: get_pure_transformer(4),
}

teacher_params = count_parameters(teacher_model)

# ===== RUNS FOR THIS NOTEBOOK =====
runs_to_do = [
    ("4-Layer-Standard", 0.3, 1.0),
    ("4-Layer-Standard", 0.3, 3.0),
    ("4-Layer-Standard", 0.3, 5.0),
    ("4-Layer-Standard", 0.5, 1.0),
    ("4-Layer-Standard", 0.5, 3.0)
]

print(f"\nRuns to complete in this notebook: {len(runs_to_do)}")
total_all = 18
completed_before = len(final_results)

for run_i, (arch_name, alpha, temperature) in enumerate(runs_to_do):
    run_key = f"{arch_name}_a{alpha}_t{temperature}"
    global_idx = completed_before + run_i + 1

    print(f"\n{'#'*70}")
    print(f"# Run {global_idx}/{total_all}: {arch_name} | alpha={alpha} | temp={temperature}")
    print(f"{'#'*70}")

    gc.collect()
    torch.cuda.empty_cache()

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    model_fn = architectures[arch_name]
    student_model = model_fn()
    student_params = count_parameters(student_model)
    print(f"Student parameters: {student_params:,} ({student_params/1e6:.1f}M)")

    is_hybrid = isinstance(student_model, DNAHybridStudent)

    args = TrainingArguments(
        output_dir=f"/kaggle/working/{run_key}",
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        fp16=True,
        logging_steps=100,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_accuracy",
        greater_is_better=True,
        save_total_limit=1,
        report_to="none",
        remove_unused_columns=not is_hybrid,
        warmup_ratio=0.1,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        learning_rate=2e-5,
    )

    start_time = time.time()

    trainer = DistillationTrainer(
        model=student_model,
        teacher_model=teacher_model,
        alpha=alpha,
        temperature=temperature,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        callbacks=[
            EarlyStoppingCallback(early_stopping_patience=3),
            ProgressCallback(),
        ],
    )

    print(f"\nStarting training...", flush=True)
    trainer.train()
    train_time = time.time() - start_time

    metrics = trainer.evaluate()
    print(f"\n--- {run_key} Results ---")
    print(f"  Accuracy:  {metrics['eval_accuracy']:.2%}")
    print(f"  F1 Score:  {metrics['eval_f1']:.4f}")
    print(f"  Precision: {metrics['eval_precision']:.4f}")
    print(f"  Recall:    {metrics['eval_recall']:.4f}")
    print(f"  Loss:      {metrics['eval_loss']:.4f}")
    print(f"  Time:      {train_time/60:.1f} min")

    save_dir = f"/kaggle/working/{run_key}_final"
    os.makedirs(save_dir, exist_ok=True)
    if is_hybrid:
        torch.save(student_model.state_dict(), os.path.join(save_dir, "model.pt"))
    else:
        student_model.save_pretrained(save_dir)

    final_results[run_key] = {
        "architecture": arch_name,
        "alpha": alpha,
        "temperature": temperature,
        "accuracy": metrics["eval_accuracy"],
        "f1": metrics["eval_f1"],
        "precision": metrics["eval_precision"],
        "recall": metrics["eval_recall"],
        "loss": metrics["eval_loss"],
        "params": student_params,
        "compression": teacher_params / student_params,
        "train_time_min": round(train_time / 60, 1),
    }

    # Save intermediate results after each run (in case of timeout)
    with open("/kaggle/working/partial_results.json", "w") as f:
        json.dump(final_results, f, indent=2)
    print(f"  Intermediate results saved ({len(final_results)} total runs)")

    del trainer, student_model
    gc.collect()
    torch.cuda.empty_cache()

print(f"\nAll {len(runs_to_do)} runs in this notebook completed!")
print(f"Total accumulated results: {len(final_results)}")


# Results Summary & Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import pandas as pd

TEACHER_ACCURACY = 0.9671  # Best teacher checkpoint accuracy

# ===== FULL RESULTS TABLE =====
print("="*100)
print("KNOWLEDGE DISTILLATION GRID SEARCH RESULTS")
print("="*100)
print(f"{'Run':<30} {'Alpha':>6} {'Temp':>6} {'Accuracy':>10} {'F1':>8} {'Params':>10} {'Compress':>10} {'Time':>8}")
print("-"*100)
print(f"{'Teacher (DNABERT-2)':<30} {'--':>6} {'--':>6} {TEACHER_ACCURACY:>9.2%} {'--':>8} {teacher_params/1e6:>8.1f}M {'1.0x':>10} {'--':>8}")
print("-"*100)

for key, res in final_results.items():
    print(f"{key:<30} {res['alpha']:>6.1f} {res['temperature']:>6.1f} {res['accuracy']:>9.2%} {res['f1']:>8.4f} {res['params']/1e6:>8.1f}M {res['compression']:>9.1f}x {res['train_time_min']:>6.1f}m")
print("="*100)

# ===== BEST CONFIG PER ARCHITECTURE =====
arch_names = list(dict.fromkeys(r["architecture"] for r in final_results.values()))
print(f"\n{'='*70}")
print("BEST CONFIGURATION PER ARCHITECTURE")
print(f"{'='*70}")
for arch in arch_names:
    arch_results = {k: v for k, v in final_results.items() if v["architecture"] == arch}
    best_key = max(arch_results, key=lambda k: arch_results[k]["accuracy"])
    best = arch_results[best_key]
    retention = best["accuracy"] / TEACHER_ACCURACY * 100
    print(f"\n  {arch}:")
    print(f"    Best config: alpha={best['alpha']}, temp={best['temperature']}")
    print(f"    Accuracy: {best['accuracy']:.2%} ({retention:.1f}% of teacher)")
    print(f"    F1: {best['f1']:.4f}")
    print(f"    Compression: {best['compression']:.1f}x")

# ===== HEATMAPS: Accuracy by Alpha x Temperature for each architecture =====
fig, axes = plt.subplots(1, len(arch_names), figsize=(7 * len(arch_names), 5))
if len(arch_names) == 1:
    axes = [axes]

for ax, arch in zip(axes, arch_names):
    arch_results = {k: v for k, v in final_results.items() if v["architecture"] == arch}
    
    heatmap_data = np.zeros((len(ALPHAS), len(TEMPERATURES)))
    for k, v in arch_results.items():
        i = ALPHAS.index(v["alpha"])
        j = TEMPERATURES.index(v["temperature"])
        heatmap_data[i, j] = v["accuracy"]
    
    im = ax.imshow(heatmap_data, cmap="YlGn", aspect="auto", vmin=0.80, vmax=1.0)
    ax.set_xticks(range(len(TEMPERATURES)))
    ax.set_xticklabels([str(t) for t in TEMPERATURES])
    ax.set_yticks(range(len(ALPHAS)))
    ax.set_yticklabels([str(a) for a in ALPHAS])
    ax.set_xlabel("Temperature")
    ax.set_ylabel("Alpha")
    ax.set_title(f"{arch}\nAccuracy Heatmap")
    
    for i in range(len(ALPHAS)):
        for j in range(len(TEMPERATURES)):
            ax.text(j, i, f"{heatmap_data[i, j]:.2%}", ha="center", va="center", fontsize=10, fontweight="bold")
    
    plt.colorbar(im, ax=ax, label="Accuracy")

plt.tight_layout()
plt.savefig("/kaggle/working/heatmap_alpha_temp.png", dpi=150, bbox_inches="tight")
plt.show()
print("Heatmap saved to /kaggle/working/heatmap_alpha_temp.png")

# ===== BAR CHART: Best config per architecture vs Teacher =====
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

best_per_arch = {}
for arch in arch_names:
    arch_results = {k: v for k, v in final_results.items() if v["architecture"] == arch}
    best_key = max(arch_results, key=lambda k: arch_results[k]["accuracy"])
    best_per_arch[arch] = arch_results[best_key]

names = list(best_per_arch.keys())
accuracies = [best_per_arch[n]["accuracy"] for n in names]
params_m = [best_per_arch[n]["params"] / 1e6 for n in names]
efficiency = [best_per_arch[n]["accuracy"] / (best_per_arch[n]["params"] / 1e6) for n in names]

colors = ["#2ecc71", "#3498db"]

# 1. Accuracy comparison
bars = axes[0].bar(names, accuracies, color=colors)
axes[0].axhline(y=TEACHER_ACCURACY, color="gold", linestyle="--", linewidth=2, label=f"Teacher ({TEACHER_ACCURACY:.2%})")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Best Student vs Teacher Accuracy")
axes[0].legend()
axes[0].set_ylim(0.75, 1.0)
for bar, acc in zip(bars, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f"{acc:.2%}", ha="center", fontsize=9)

# 2. Model size
all_names = ["Teacher"] + names
all_params = [teacher_params / 1e6] + params_m
all_colors = ["gold"] + colors
bars2 = axes[1].bar(all_names, all_params, color=all_colors)
axes[1].set_ylabel("Parameters (M)")
axes[1].set_title("Model Size Comparison")
for bar, p in zip(bars2, all_params):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f"{p:.1f}M", ha="center", fontsize=9)

# 3. Efficiency
bars3 = axes[2].bar(names, efficiency, color=colors)
axes[2].set_ylabel("Accuracy / M params")
axes[2].set_title("Efficiency (Accuracy per M Parameters)")
for bar, e in zip(bars3, efficiency):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0002, f"{e:.4f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("/kaggle/working/distillation_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Charts saved to /kaggle/working/distillation_results.png")

# ===== SAVE RESULTS JSON =====
results_json = {
    "teacher_accuracy": TEACHER_ACCURACY,
    "teacher_params": teacher_params,
    "alphas": ALPHAS,
    "temperatures": TEMPERATURES,
    "runs": final_results,
}
with open("/kaggle/working/distillation_results.json", "w") as f:
    json.dump(results_json, f, indent=2)
print("Results saved to /kaggle/working/distillation_results.json")